# 🔬 Krypto: Mobile Forensic Activity Reconstruction Fine-Tuning
### Fine-Tuning Lightweight LLMs (Gemma-2-2B / Gemma-4-E2B) with Unsloth on H100/A100 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmanuelbadmus/Krypto/blob/main/FineTune_Gemma_Forensics.ipynb)

---
### 🎯 Objectives
1. **100% Citation Discipline**: Eliminate hallucinations by enforcing strict `[EVT-xxxxxxxxxxxx]` ground-truth provenance citations.
2. **Absence & SQLCipher Auditing**: Teach the model to explicitly document unrecoverable/encrypted data (Signal, Google Podcasts, WeChat residue) rather than omitting them.
3. **Multi-Signal Indirect Commute Deduction**: Synthesize unlogged Android Auto commutes from Bluetooth and charging power logs.
4. **Beat Non-AI Baseline**: Outperform the 68.2% baseline recall in `baseline_benchmark_report.txt`.

## 🛠️ Step 1: Install Unsloth & Dependencies

In [ ]:
# Install Unsloth for 2x faster training and 70% lower VRAM usage
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install torch transformers datasets trl peft accelerate bitsandbytes sentencepiece protobuf scikit-learn tabulate

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")

## 📂 Step 2: Clone Repository or Authenticate (Private Repo Support)

In [ ]:
import os
import sys

# Check if running inside Google Colab
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB and not os.path.exists("data/splits/train.jsonl") and not os.path.exists("Krypto/data/splits/train.jsonl"):
    print("Cloning Krypto repository...")
    # If the GitHub repository is private, you can paste your GitHub Personal Access Token (PAT) when prompted
    # Or press Enter if public
    try:
        from getpass import getpass
        token = getpass("Enter GitHub Personal Access Token (or press Enter if public): ").strip()
        if token:
            !git clone https://{token}@github.com/emmanuelbadmus/Krypto.git
        else:
            !git clone https://github.com/emmanuelbadmus/Krypto.git
    except Exception as e:
        !git clone https://github.com/emmanuelbadmus/Krypto.git

# Automatically switch directory into Krypto root if cloned
if os.path.exists("Krypto/data/splits/train.jsonl"):
    os.chdir("Krypto")
    print(f"Changed directory to: {os.getcwd()}")

# Locate dataset files automatically
def resolve_data_path(rel_path):
    candidates = [
        rel_path,
        os.path.join("Krypto", rel_path),
        os.path.join("/content", rel_path),
        os.path.join("/content/Krypto", rel_path),
    ]
    for c in candidates:
        if os.path.exists(c):
            return os.path.abspath(c)
    return rel_path

train_file = resolve_data_path("data/splits/train.jsonl")
val_file = resolve_data_path("data/splits/val.jsonl")

print(f"\n[Data Path Check]")
print(f"  Train File: {train_file} -> Exists: {os.path.exists(train_file)}")
print(f"  Val File:   {val_file}   -> Exists: {os.path.exists(val_file)}")

if os.path.exists(train_file):
    with open(train_file) as f:
        print(f"  Train Samples (Date-Held-Out): {len(f.readlines())} windows")
if os.path.exists(val_file):
    with open(val_file) as f:
        print(f"  Val Benchmark Samples:         {len(f.readlines())} windows")
else:
    print("\n[Notice] If files are missing, you can upload 'train.jsonl' and 'val.jsonl' directly to the files panel on the left!")

## 🧠 Step 3: Load Base Model with Unsloth FastLanguageModel

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/gemma-2-2b-it" # or "google/gemma-4-E2B-it" / "unsloth/Qwen2.5-14B-Instruct"
MAX_SEQ_LENGTH = 8192              # 8k context for long SQLite dumps
LOAD_IN_4BIT = False               # False for full bfloat16 on H100/A100; True for QLoRA on smaller GPUs

print(f"Loading {MODEL_NAME} with FastLanguageModel...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma")
print("Base model and tokenizer loaded successfully!")

## 💉 Step 4: Inject LoRA Target Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                # LoRA Rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,       # LoRA Alpha scaling
    lora_dropout=0.0,    # 0.0 optimized for Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## 📊 Step 5: Format Datasets with Chat Template

In [ ]:
import json
from datasets import Dataset

def prepare_dataset(filepath, tokenizer):
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            msgs = item.get("messages", [])
            if len(msgs) >= 2:
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                samples.append({"text": text})
    return Dataset.from_list(samples)

train_dataset = prepare_dataset(train_file, tokenizer)
val_dataset = prepare_dataset(val_file, tokenizer) if os.path.exists(val_file) else None

print(f"Formatted {len(train_dataset)} training samples.")
if val_dataset:
    print(f"Formatted {len(val_dataset)} validation samples.")

## 🚀 Step 6: Train Model with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "outputs_forensic_gemma_2b"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 8
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    logging_steps=1,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Starting GPU Fine-Tuning...")
trainer_stats = trainer.train()
print(f"Training Complete! Runtime: {trainer_stats.metrics['train_runtime']:.2f}s")

## 💾 Step 7: Save Fine-Tuned LoRA Adapter

In [ ]:
final_adapter_dir = f"{OUTPUT_DIR}/final_adapter"
print(f"Saving LoRA adapter to {final_adapter_dir}...")
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print("Saved adapter successfully!")

## 🏆 Step 8: Automated Benchmark Evaluation on Held-Out Windows

In [ ]:
gt_file = resolve_data_path("data/raw_database/ground_truth.csv")
events_db_file = resolve_data_path("data/raw_database/events_db.jsonl")

!python3 -m harness.runner \
  --model "unsloth/gemma-2-2b-it" \
  --adapter "outputs_forensic_gemma_2b/final_adapter" \
  --data "{val_file}" \
  --ground_truth "{gt_file}" \
  --events_db "{events_db_file}" \
  --output_json "eval_reports/eval_finetuned_gemma_val.json" \
  --output_md "eval_reports/eval_finetuned_gemma_val.md" \
  --max_new_tokens 350

# Display Markdown Report
from IPython.display import display, Markdown
if os.path.exists("eval_reports/eval_finetuned_gemma_val.md"):
    with open("eval_reports/eval_finetuned_gemma_val.md") as f:
        display(Markdown(f.read()))

## 🔍 Step 9: Live Interactive Reconstruction Test

In [ ]:
FastLanguageModel.for_inference(model)

# Test sample prompt
test_prompt = """Date: 2019-03-15
EXTRACTED ARTIFACTS:
[EVT-a9668b712bee] 17:33 Twitter tweet_seen_or_posted <224423919>
[EVT-b86eea5bdfcc] 08:03 Twitter tweet_seen_or_posted <804341497441255424>
PROVENANCE:
EVT-a9668b712bee -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 135
EVT-b86eea5bdfcc -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 488

Reconstruct the user activity for this window."""

messages = [
    {"role": "system", "content": "You are a digital forensic analyst. Reconstruct the chronological user activity from the extracted Android artifacts. Cite evidence using exact [EVT-xxxx] IDs. Explicitly state unrecoverable apps."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.inference_mode():
    outputs = model.generate(inputs, max_new_tokens=300, temperature=0.1, use_cache=True)

print("=== RECONSTRUCTED OUTPUT ===")
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))